# H1: Scale Correspondence Validation

Validates Hypothesis H1: ξ(k) = ξ₀ · exp(-k/k_c) with R² > 0.95.



In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import torch

from src.utils.seed_registry import SeedRegistry
from src.utils.device_manager import DeviceManager

SeedRegistry.get_instance().set_master_seed(42)
device = DeviceManager.get_instance().get_device()
print(f'Device: {device}')


## 1. Correlation Length Measurement

For each layer k, compute ξ(k) from Fisher spectrum.

In [ ]:
from src.core.correlation.estimators import ExponentialDecayFitter
from src.core.correlation.two_point import chi1_gauss_hermite

# Compute chi1 at critical init (sigma_w=1.4, sigma_b=0.3)
sigma_w2 = 1.4**2
chi1 = chi1_gauss_hermite(sigma_w2, nonlinearity="tanh")
print(f"chi1 at (sigma_w=1.4, sigma_b=0.3): {chi1:.6f}")
print(f"xi_depth = -1/log(chi1) = {-1/np.log(max(chi1, 1e-10)):.1f}")


## 2. Exponential Decay Fit

In [ ]:
# Generate synthetic correlation lengths for demonstration
from src.core.correlation.exponential_decay_fitter import ExponentialDecayFitter as EDF

rng = np.random.default_rng(42)
k_arr = np.arange(30, dtype=float)
xi_true = 20.0 * np.exp(-k_arr / 8.0)
xi_noisy = xi_true + rng.normal(0, 0.1, 30) * xi_true

fitter = EDF(p0_xi0=xi_noisy[0], p0_kc=8.0)
result = fitter.fit(k_arr, xi_noisy)
print(f"Fitted: xi_0={result.xi_0:.2f}, k_c={result.k_c:.2f}, R²={result.r2:.4f}")
print(f"chi1 from fit: {result.chi1:.4f}")

assert result.r2 > 0.95, f"R²={result.r2:.4f} below threshold 0.95"
print("PASS: R² > 0.95")


## 3. Reproduce Figure 3 (H1 panels)

In [ ]:
import subprocess, sys
result = subprocess.run([
    sys.executable, "../figures/manuscript/generate_figure3.py",
    "--fast-track", "--output", "/tmp/fig3_notebook.pdf"
], capture_output=True, text=True)
print(result.stdout)
if result.returncode == 0:
    print("Figure 3 generated successfully")
